# MicroCLIP — Colab Evaluation

Zero-shot classification (CIFAR-10/100) and COCO 5K retrieval for a trained checkpoint.

Any GPU runtime works (eval is cheap — an A100 is overkill but fine).
Checkpoints are read from the same Drive folder the training notebook writes to.
Only `val2017` is downloaded (~1 GB); CIFAR downloads itself.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PERSIST = "/content/drive/MyDrive/microclip"

In [ ]:
%cd /content
!git clone https://github.com/umutonuryasar/microclip.git 2>/dev/null || git -C microclip pull
%cd /content/microclip
!pip -q install -e .

In [ ]:
import os

# Reuse checkpoints + tokenizer written by the training notebook.
for link, target in [("runs", f"{PERSIST}/runs"), ("artifacts", f"{PERSIST}/artifacts")]:
    if not os.path.islink(link):
        os.system(f"rm -rf {link}")
        os.symlink(target, link)
!ls runs/

In [ ]:
%%bash
# Retrieval needs val2017 + annotations only.
mkdir -p data/coco && cd data/coco
if [ ! -d val2017 ]; then
  wget -q -c http://images.cocodataset.org/zips/val2017.zip
  unzip -q val2017.zip && rm val2017.zip
fi
if [ ! -d annotations ]; then
  wget -q -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip
  unzip -q annotations_trainval2017.zip && rm annotations_trainval2017.zip
fi

## Evaluate

`CONFIG` must be the config the checkpoint was trained with.
Results are printed as JSON and also saved under `Drive/microclip/results/`.

In [ ]:
RUN = "sigmoid_b512"
CHECKPOINT = f"runs/{RUN}/best.pt"
CONFIG = f"configs/{RUN}.yml"  # ablations live under configs/ablations/

import os
os.makedirs(f"{PERSIST}/results", exist_ok=True)

In [ ]:
!python scripts/evaluate.py --checkpoint $CHECKPOINT --config $CONFIG --task zeroshot \
    | tee $PERSIST/results/{RUN}_zeroshot.json

In [ ]:
!python scripts/evaluate.py --checkpoint $CHECKPOINT --config $CONFIG --task retrieval \
    | tee $PERSIST/results/{RUN}_retrieval.json